### Middleware

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


#### Summarization Middleware

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="groq:qwen/qwen3.8-27b",
        trigger=("messages", 10),
        keep=("messages", 4)
    )]
)

In [9]:
### run with thread id
config = {"configurable": {"thread_id": "test-1"}}

In [10]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='f8a415cb-ad93-472b-9330-ef15aa96e0e5'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 19, 'total_tokens': 27, 'completion_time': 0.01801677, 'completion_tokens_details': None, 'prompt_time': 0.001487259, 'prompt_tokens_details': None, 'queue_time': 0.021540598, 'total_time': 0.019504029}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_43dd1ea4ee', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16d9c-c9b7-73af-82f0-d2b35cf9e3d1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_tokens': 8, 'total_tokens': 27})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='f8a415cb-ad93-472b-9330-ef15aa96e0e5'), AIMessage(conte

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.8-27b",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

#### Human In the Loop MiddleWare

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """mock function to read an email by its ID."""
    return f"email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """mock function to send an email."""
    return f"email sent to {recipient} with subject '{subject}'"

In [14]:
agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "send_email_tool": {
                "allowed_decisions": ["approve", "edit", "reject"]
            },
            "read_email_tool": False
        }
    )]
)

In [15]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [16]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='48e2997d-6a89-414d-8984-778082477177'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '4mf902dhh', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 376, 'total_tokens': 433, 'completion_time': 0.178915236, 'completion_tokens_details': None, 'prompt_time': 0.027370912, 'prompt_tokens_details': None, 'queue_time': 0.107087053, 'total_time': 0.206286148}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_8999620a6e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16daa-e5e9-709d-8fd1-edc1dd6975f8-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': '

In [17]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: Email sent to john@test.com with subject "Hello" and body "How are you?" ✅


In [18]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='48e2997d-6a89-414d-8984-778082477177'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '4mf902dhh', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 376, 'total_tokens': 433, 'completion_time': 0.178915236, 'completion_tokens_details': None, 'prompt_time': 0.027370912, 'prompt_tokens_details': None, 'queue_time': 0.107087053, 'total_time': 0.206286148}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_8999620a6e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16daa-e5e9-709d-8fd1-edc1dd6975f8-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': '

In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [21]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [22]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I attempted to send the email to john@test.com with the subject "Hello" and body "How are you?", but the action was rejected and not executed.

If you'd like me to try again or make any changes to the email, just let me know.


In [23]:

result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='19932e5f-2602-46a2-8d18-f0f61277c32e'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'q75hrqjyb', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 376, 'total_tokens': 433, 'completion_time': 0.250102979, 'completion_tokens_details': None, 'prompt_time': 0.045343529, 'prompt_tokens_details': None, 'queue_time': 0.021777638, 'total_time': 0.295446508}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_8999620a6e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16dae-8516-7572-8eb9-1bbfcfde9bc4-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': '

In [24]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [25]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [26]:

result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='6f46169d-2ec9-4e35-9c40-bab274e561d2'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9gd3jc34c', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 374, 'total_tokens': 428, 'completion_time': 0.163186471, 'completion_tokens_details': None, 'prompt_time': 0.027568239, 'prompt_tokens_details': None, 'queue_time': 0.060073174, 'total_time': 0.19075471}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_8999620a6e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16daf-5028-77ab-a7dc-6d0ef5d81be0-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'Hello', 'reci

In [27]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: The email was sent, but a human reviewer modified it before it went out. It was delivered to **correct@email.com** with the subject **"Corrected Subject"** and body **"This was edited by human before sending"** instead of the original details you provided.

Let me know if you'd like me to do anything else.


In [28]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='6f46169d-2ec9-4e35-9c40-bab274e561d2'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9gd3jc34c', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 374, 'total_tokens': 428, 'completion_time': 0.163186471, 'completion_tokens_details': None, 'prompt_time': 0.027568239, 'prompt_tokens_details': None, 'queue_time': 0.060073174, 'total_time': 0.19075471}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_8999620a6e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16daf-5028-77ab-a7dc-6d0ef5d81be0-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'Hello', 'reci